# MapleMind - QLoRA SFT on a free Colab T4
Runtime -> Change runtime type -> T4 GPU. Expected: ~1.5B params in 4-bit, ~5-7 GB VRAM peak (estimate - the script logs the measured value to `train_metrics.json`).

In [ ]:
!git clone <your-repo-url> maplemind
%cd maplemind
!pip -q install -r requirements.txt -r finetune/requirements-train.txt
!pip -q install -e .

In [ ]:
# 1) Build leakage-safe SFT data (KB-grounded + abstention + boundaries + MentalChat16K empathy subset)
!python finetune/build_sft_dataset.py --mentalchat 400 --abstain-rate 0.2
!cat finetune/data/stats.json

In [ ]:
# 2) Smoke test (5 steps), then full training
!python finetune/train_lora.py --max-steps 5
!python finetune/train_lora.py

In [ ]:
import json; m=json.load(open('finetune/outputs/maplemind-qwen25-1.5b-lora/train_metrics.json')); print({k:v for k,v in m.items() if k!='log_history'})

In [ ]:
# 3) Held-out comparison: base vs fine-tuned vs open alternative, with and without RAG
!python eval/model_matrix.py --models base_qwen25_1.5b ft_qwen25_1.5b alt_llama32_3b --split test
!cat eval/reports/model_matrix_test.md

In [ ]:
# 4) End-to-end harness with the fine-tuned model inside the LangGraph app
import os
os.environ['MAPLEMIND__LLM__PROVIDER']='hf'
!python -m maplemind.cli ask 'Is depression linked to obesity?' --adapter

Download `finetune/outputs/maplemind-qwen25-1.5b-lora` (adapter only, ~70 MB) and `eval/reports/` for the presentation.